# 02 — Exploration des schémas de placement

Phase 2 : on compare les 5 schémas de placement de capteurs sur le même
champ. Pour chaque schéma on visualise les positions, on calcule la
prévalence observée, et on regarde la distance moyenne au capteur le plus
proche depuis chaque plant (couverture spatiale).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from aphid_spatial.simulation import (
    Field,
    FieldConfig,
    SensorConfig,
    place_sensors,
    simulate_field,
)
from aphid_spatial.visualization.maps import plot_sensors

FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

data_path = Path("../data/simulated/field_default.npz")
if data_path.exists():
    field = Field.load(data_path)
else:
    field = simulate_field(FieldConfig(seed=42))

In [ ]:
schemes = ["uniform", "grid", "stratified", "edge_biased", "poisson_disk"]
all_readings = {}
for s in schemes:
    all_readings[s] = place_sensors(
        field, SensorConfig(n_sensors=20, placement=s, seed=2024)
    )

In [ ]:
fig, axes = plt.subplots(len(schemes), 1, figsize=(14, 2.7 * len(schemes)), constrained_layout=True)
for ax, s in zip(axes, schemes, strict=True):
    plot_sensors(all_readings[s], field, ax=ax, title=f"Schéma : {s}")
fig.savefig(FIG_DIR / "02_exploration_schemes.png", dpi=150)
plt.show()

## Statistiques par schéma

- **prévalence_obs** : fraction de capteurs avec ``obs=1`` (à comparer à
  la prévalence vraie sur tout le champ).
- **dist_moyenne** : distance moyenne, depuis chaque plant, au capteur le
  plus proche (en mètres). Plus c'est petit, mieux le champ est couvert.

In [ ]:
rows = []
for s, r in all_readings.items():
    tree = cKDTree(r.coords)
    dist, _ = tree.query(field.coords, k=1)
    rows.append(
        {
            "scheme": s,
            "prevalence_obs": float(r.obs.mean()),
            "n_positifs": int(r.obs.sum()),
            "dist_moyenne_m": float(dist.mean()),
            "dist_max_m": float(dist.max()),
        }
    )
df = pd.DataFrame(rows)
df.insert(1, "prevalence_vraie", float(field.presence.mean()))
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(df["scheme"], df["prevalence_obs"], color="steelblue")
axes[0].axhline(float(field.presence.mean()), color="red", linestyle="--", label="prévalence vraie")
axes[0].set_ylabel("prévalence observée")
axes[0].set_title("Détection vs prévalence vraie")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(df["scheme"], df["dist_moyenne_m"], color="darkorange")
axes[1].set_ylabel("distance moyenne au capteur (m)")
axes[1].set_title("Couverture spatiale")
axes[1].tick_params(axis="x", rotation=20)
fig.savefig(FIG_DIR / "02_exploration_stats.png", dpi=150)
plt.show()